In [1]:
import requests
import markdown
from bs4 import BeautifulSoup
from openai import OpenAI
import chromadb
from chromadb.utils import embedding_functions
from langchain_text_splitters import RecursiveCharacterTextSplitter
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output
import pandas as pd
import getpass
import os

In [2]:
display(HTML("<h3>🔑 System Initialization</h3>"))
api_key = getpass.getpass("Please enter your Avalai API Key: ")
client = OpenAI(api_key=api_key, base_url="https://api.avalai.ir/v1")

display(HTML("<p>⏳ <i>Loading Multilingual Embedding Model...</i></p>"))
multilingual_emb = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name="paraphrase-multilingual-MiniLM-L12-v2"
)

chroma_client = chromadb.Client()
collection_name = "interactive_pro_rag"
try:
    chroma_client.delete_collection(name=collection_name)
except:
    pass
collection = chroma_client.create_collection(name=collection_name, embedding_function=multilingual_emb)

# History Storage
history_data = []
print('\n')
display(HTML("<p style='color: green;'>✅ <b>System Ready! Run Cell 2 to start the UI.</b></p>"))

Please enter your Avalai API Key:  ········


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


---

In [3]:
# Core Engine Functions
def advanced_web_scraper(url):
    headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) Chrome/120.0.0.0 Safari/537.36'}
    try:
        response = requests.get(url, headers=headers, timeout=15)
        response.encoding = 'utf-8' 
        response.raise_for_status()
        
        soup = BeautifulSoup(response.text, 'html.parser')
        for junk in soup(["script", "style", "nav", "footer", "header", "aside", "form"]):
            junk.decompose()
            
        text_elements = soup.find_all(['p', 'h1', 'h2', 'h3', 'li', 'td', 'th'])
        full_text = " ".join([tag.get_text(strip=True) for tag in text_elements if len(tag.get_text(strip=True)) > 20])
        
        if not full_text:
            return [], "No meaningful text found."

        text_splitter = RecursiveCharacterTextSplitter(chunk_size=800, chunk_overlap=150)
        chunks = text_splitter.split_text(full_text)
        return chunks, None
    except Exception as e:
        return [], f"Scraping Error: {str(e)}"

def generate_rag_answer(query, language):
    results = collection.query(query_texts=[query], n_results=10)
    candidates = results['documents'][0] if results['documents'] else []
    
    if not candidates:
        return "No relevant context found in the database."

    top_context = "\n---\n".join(candidates)
    
    if language == "فارسی (FA)":
        sys_prompt = "شما یک دستیار هوشمند و دقیق هستید. پاسخ کاربر را منحصراً و کامل بر اساس 'متن منبع' بدهید. اگر اطلاعات پراکنده است آن‌ها را مرتب کنید. اگر در متن نیست بگویید اطلاعات یافت نشد."
        user_prompt = f"متن منبع:\n{top_context}\n\nسوال کاربر: {query}"
    else:
        sys_prompt = "You are a precise AI assistant. Answer strictly and comprehensively using the 'Source Context'. If not present, state it."
        user_prompt = f"Source Context:\n{top_context}\n\nUser Question: {query}"

    try:
        response = client.chat.completions.create(
            model="gpt-4o", 
            messages=[{"role": "system", "content": sys_prompt}, {"role": "user", "content": user_prompt}],
            temperature=0.1
        )
        return response.choices[0].message.content
    except Exception as e:
        return f"LLM Error: {str(e)}"

In [5]:
# Interactive Widget UI

# Injecting Custom Web Fonts and Global CSS
display(HTML("""
<style>
@import url('https://fonts.googleapis.com/css2?family=Baloo+Bhaijaan+2:wght@400;600;800&family=Lalezar&display=swap');

/* Master classes for RTL/LTR consistency */
.rtl-container {
    direction: rtl !important;
    text-align: right !important;
    font-family: 'Baloo Bhaijaan 2', sans-serif !important;
    width: 100%;
}
.rtl-heading {
    font-family: 'Lalezar', cursive !important;
    color: #198754;
    direction: rtl !important;
    text-align: right !important;
}
.ltr-container {
    direction: ltr !important;
    text-align: left !important;
    font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif !important;
}

/* Fix Markdown rendering inside RTL containers */
.markdown-rtl ul, .markdown-rtl ol {
    padding-right: 25px !important;
    padding-left: 0 !important;
    margin-right: 15px !important;
}
.markdown-rtl li {
    margin-bottom: 8px !important;
    line-height: 1.8 !important;
}
.markdown-rtl p {
    line-height: 1.8 !important;
    margin-bottom: 12px !important;
}
.markdown-rtl strong {
    font-family: 'Baloo Bhaijaan 2', sans-serif !important;
    font-weight: 800 !important;
    color: #0056b3 !important; /* Make bold text pop */
}
</style>
"""))

# Widgets Definition
url_input = widgets.Text(description='🔗 URL:', placeholder='Enter website URL here...', layout=widgets.Layout(width='80%'))
btn_scrape = widgets.Button(description='Extract Data', button_style='primary')
box_url = widgets.HBox([url_input, btn_scrape])

lang_dropdown = widgets.Dropdown(options=['فارسی (FA)', 'English (EN)'], value='فارسی (FA)', description='🗣️ Language:')
question_input = widgets.Textarea(description='❓ Question:', placeholder='Type your question...', layout=widgets.Layout(width='80%', height='80px'))
btn_ask = widgets.Button(description='Ask LLM', button_style='success', layout=widgets.Layout(height='80px'))
box_question = widgets.HBox([lang_dropdown, question_input, btn_ask])
box_question.layout.display = 'none' 

btn_refresh = widgets.Button(description='🔄 Reset Session', button_style='warning')
btn_export = widgets.Button(description='💾 Export CSV', button_style='info')
box_actions = widgets.HBox([btn_refresh, btn_export])
box_actions.layout.display = 'none' 

out_chunks = widgets.Output()
out_answer = widgets.Output()

# State
current_url_chunks = []

# HTML/CSS Helpers
def format_chunks_html(chunks):
    html = f"<div class='rtl-container'><h4 class='rtl-heading' style='color:#0d6efd;'>✅ استخراج {len(chunks)} قطعه متن:</h4>"
    # Added fixed height and scroll matching the chunk box requested
    html += "<div style='max-height: 350px; overflow-y: auto; border: 2px solid #e0e0e0; padding: 15px; background-color: #f8f9fa; border-radius: 10px; margin-bottom: 20px; box-shadow: inset 0 2px 4px rgba(0,0,0,0.05);'>"
    for i, chunk in enumerate(chunks):
        is_rtl = any("\u0600" <= c <= "\u06FF" for c in chunk[:50])
        container_class = "rtl-container" if is_rtl else "ltr-container"
        
        html += f"<div style='margin-bottom: 15px; padding-bottom: 15px; border-bottom: 1px dashed #ccc;'>"
        html += f"<b style='color: #dc3545; font-family: Tahoma;'>[Chunk {i+1}]</b><br>"
        html += f"<div class='{container_class}' style='font-size: 15px; color: #333; line-height: 1.8; margin-top: 5px;'>{chunk}</div></div>"
    html += "</div></div>"
    return html

def format_answer_html(answer, lang):
    """Converts Markdown to HTML and applies perfect structural styling."""
    is_persian = lang == "فارسی (FA)"
    container_class = "rtl-container markdown-rtl" if is_persian else "ltr-container"
    heading_class = "rtl-heading" if is_persian else ""
    heading_text = "✨ پاسخ هوش مصنوعی" if is_persian else "✨ AI Response"
    
    # Safely convert Markdown from the LLM into clean HTML
    html_content = markdown.markdown(answer, extensions=['extra'])
    
    # Exact same dimensional styling as the chunk box
    return f"""
    <div style='border: 2px solid #198754; border-radius: 10px; padding: 15px; background-color: #f1fdf6; margin-top: 15px; margin-bottom: 20px; box-shadow: 0 4px 6px rgba(0,0,0,0.05);'>
        <h3 class='{heading_class}' style='margin-top: 0; margin-bottom: 10px; font-size: 1.5rem;'>{heading_text}</h3>
        <div class='{container_class}' style='max-height: 350px; overflow-y: auto; padding-right: 10px;'>
            <div style='font-size: 16px; color: #1a1a1a;'>{html_content}</div>
        </div>
    </div>
    """

def reset_database():
    global collection
    try:
        chroma_client.delete_collection(name=collection_name)
    except:
        pass
    collection = chroma_client.create_collection(name=collection_name, embedding_function=multilingual_emb)

# Event Handlers
def on_scrape_click(b):
    global current_url_chunks
    box_question.layout.display = 'none'
    box_actions.layout.display = 'none'
    out_answer.clear_output()
    
    with out_chunks:
        clear_output()
        url = url_input.value.strip()
        if not url:
            display(HTML("<p style='color:red;'>Please enter a URL.</p>"))
            return
            
        display(HTML(f"<p style='font-family: Tahoma;'>⏳ <i>Fetching and processing {url} ...</i></p>"))
        chunks, err = advanced_web_scraper(url)
        
        if err:
            display(HTML(f"<p style='color:red;'>❌ {err}</p>"))
            return
            
        current_url_chunks = chunks
        reset_database()
        collection.add(documents=chunks, ids=[f"chk_{i}" for i in range(len(chunks))])
        
        clear_output()
        display(HTML(format_chunks_html(chunks)))
        box_question.layout.display = 'flex'

def on_ask_click(b):
    box_actions.layout.display = 'none'
    
    with out_answer:
        clear_output()
        query = question_input.value.strip()
        lang = lang_dropdown.value
        url = url_input.value.strip()
        
        if not query:
            display(HTML("<p style='color:red;'>Please enter a question.</p>"))
            return
            
        display(HTML(f"<p style='font-family: Tahoma;'>🤖 <i>Analyzing query against {len(current_url_chunks)} chunks...</i></p>"))
        
        answer = generate_rag_answer(query, lang)
        clear_output()
        display(HTML(format_answer_html(answer, lang)))
        
        history_data.append({"URL": url, "Language": lang, "Question": query, "Answer": answer})
        box_actions.layout.display = 'flex'

def on_refresh_click(b):
    out_chunks.clear_output()
    out_answer.clear_output()
    box_question.layout.display = 'none'
    box_actions.layout.display = 'none'
    url_input.value = ''
    question_input.value = ''
    global current_url_chunks
    current_url_chunks = []
    reset_database()
    with out_chunks:
        display(HTML("<p style='color:orange; font-family: Tahoma;'>🔄 Session reset. Ready for a new URL.</p>"))

def on_export_click(b):
    with out_answer:
        if not history_data:
            display(HTML("<p style='color:red;'>No history to export yet.</p>"))
            return
        df = pd.DataFrame(history_data)
        os.makedirs("exports", exist_ok=True)
        filename = "exports/rag_history.csv"
        df.to_csv(filename, index=False, encoding="utf-8-sig") 
        display(HTML(f"<p style='color:green; font-family: Tahoma;'>💾 <b>History successfully saved to:</b> <code>{filename}</code></p>"))

# Bind Events
btn_scrape.on_click(on_scrape_click)
btn_ask.on_click(on_ask_click)
btn_refresh.on_click(on_refresh_click)
btn_export.on_click(on_export_click)

# Display UI
display(HTML("<h2 style='font-family: Tahoma; color: white;'>🌐 Universal Agentic RAG System</h2>"))
display(box_url)
display(out_chunks)      
display(box_question)    
display(out_answer)      
display(box_actions)

Output()

Output()